# Transformer(8. Feed Forward Network(FFN)) 언어 모델 총정리 

## 1. Decoder Block
	◦ 트랜스포머 디코더 블록 단계별 공식
	1) 입력값(token) 
	2) 임베딩 + 위치인코딩 
	3) Masked Multi-Head Attention 
	4) 1차 Residual Connection → 1차 Layer Normalization
	5) Cross Attention(인코더 Key/Value, 디코더 Query 사용하여 Attention 계산) 
	6) 2차 Residual Connection → 2차 Layer Normalization 
	7) Feed Forward Network(FFN)
	8) 3차 Residual Connection → 3차 Layer Normalization
	9) Linear Layer(선형계층)
	10) Softmax(소프트맥스 확률분포)

---
### 1) 임베딩 + 위치 인코딩
$ [ Y = \text{Embedding}(tokens) + \text{PositionalEncoding} ] $

### 2) Masked Multi-Head Attention
$ [ \text{MaskedMHA}(Y) = \text{Softmax}\left(\frac{QK^\top}{\sqrt{d_k}} + \text{mask}\right)V ] $

### 3) Residual Connection + Layer Normalization (1차)
$ [ H_1 = \text{LayerNorm}(Y + \text{MaskedMHA}(Y)) ] $

### 4) Cross Attention (인코더 Key/Value, 디코더 Query 사용하여 Attention 계산)
$ [ \text{CrossAttn}(H_1, H_{enc}) = \text{Softmax}\left(\frac{Q_{dec}K_{enc}^\top}{\sqrt{d_k}}\right)V_{enc} ] $

### 5) Residual Connection + Normalization (2차)
$ [ H_2 = \text{LayerNorm}(H_1 + \text{CrossAttn}(H_1, H_{enc})) ] $

### 6) Feed Forward Network (FFN)
$ [ \text{FFN}(H_2) = \max(0, H_2 W_1 + b_1) W_2 + b_2 ] $

### 7) Residual Connection + LayerNorm (3차)
$ [ H_3 = \text{LayerNorm}(H_2 + \text{FFN}(H_2)) ] $

### 8) Linear Layer(선형계층)
$ [ z = x W_o + b_o ] $

### 9) Softmax(소프트맥스 확률분포)
$ [ P_i = \frac{e^{z_i}}{\sum_{j=1}^{V} e^{z_j}} ] $

---

## 2. Transformer(8. Feed Forward Network(FFN)) 언어 모델의 Feed Forward Network(FFN) 계산 과정을 공식과 함께 단계별로 계산 
---
### 1) 입력값, 초기 가중치 편향값 설정
	◦ 공식 :
$ [ h = x W_1 + b_1 ] $

입력값)

Cross-Attention 이후 2차 Residual + LayerNorm 결과:
$ [ x = [-1.0, 1.0] ] $

가중치 및 편향 초기 설정값)

$ 차원 확장 : [ d=2 \to d_{ff}=4 ]
[ W_1 =
\begin{bmatrix}
0.2 & -0.3 & 0.5 & 0.1 \\
0.4 & 0.1 & -0.2 & 0.3
\end{bmatrix}, \quad
b_1 = [0.1, -0.1, 0.2, 0.0] ] $

$ 차원 축소 : [ d_{ff}=4 \to d=2 ]
[ W_2 =
\begin{bmatrix}
0.3 & -0.2 \\
-0.4 & 0.1 \\
0.5 & 0.2 \\
0.1 & -0.3
\end{bmatrix}, \quad
b_2 = [0.0, 0.0] ] $

---
### 2) 1차 선형 변환 (차원 확장)
$ [ h = x W_1 + b_1 ] $

가중치 계산 :

$ h = [-1.0, 1.0] \cdot
\begin{bmatrix}
0.2 & -0.3 & 0.5 & 0.1 \\
0.4 & 0.1 & -0.2 & 0.3
\end{bmatrix} $

$ = [(-1)(0.2)+(1)(0.4), (-1)(-0.3)+(1)(0.1), (-1)(0.5)+(1)(-0.2), (-1)(0.1)+(1)(0.3)] $

$ = [0.2, 0.4, -0.7, 0.2] $

편향 추가 :

$ h = [0.2+0.1, 0.4-0.1, -0.7+0.2, 0.2+0.0] $ 

$ = [0.3, 0.3, -0.5, 0.2] $

---
### 3) ReLU(비선형 함수) 적용 : 음수는 0으로 잘라내고 양수만 남김, 복잡한 패턴 학습 가능
$ [ h^\prime = \max(0, h) = [0.3, 0.3, 0, 0.2] ] $

---
### 4) 2차 선형 변환 (차원 축소)
$ [ y = h^\prime W_2 + b_2 ] $

계산:

$ [ [0.3, 0.3, 0, 0.2] \cdot
\begin{bmatrix}
0.3 & -0.2 \\
-0.4 & 0.1 \\
0.5 & 0.2 \\
0.1 & -0.3
\end{bmatrix}
 ] $

첫 번째 성분 :
$ [ (0.3)(0.3) + (0.3)(-0.4) + (0)(0.5) + (0.2)(0.1) = 0.09 - 0.12 + 0 + 0.02 = -0.01 ] $

두 번째 성분 :
$ [ (0.3)(-0.2) + (0.3)(0.1) + (0)(0.2) + (0.2)(-0.3) = -0.06 + 0.03 + 0 - 0.06 = -0.09 ] $

따라서:
$ [ y = [-0.01, -0.09] ] $

---